# Phase 4 — Optimisation des hyperparamètres du XGBoost (FD001)

Applique les sections 6 et 8 de [`docs/protocole_evaluation.md`](../docs/protocole_evaluation.md).

**Question** : les réglages actuels (`n_estimators=200, max_depth=5, learning_rate=0.05`) n'ont jamais été cherchés. Une recherche systématique trouve-t-elle des réglages **significativement** meilleurs ?

**Plan**
1. Mise en place : mêmes données, mêmes 46 features, mêmes plis par moteur que le notebook 03 (§1 du protocole).
2. **Règle de décision, écrite avant tout résultat de recherche** (§8).
3. Trois méthodes à **budget égal** : `GridSearchCV`, `RandomizedSearchCV`, Optuna (TPE, graine 42).
4. Biais d'optimisme : pourquoi le meilleur de N essais est trop beau.
5. Confirmation sur des **plis neufs** (autres graines de découpage) : test t apparié et Wilcoxon, meilleur réglage contre réglage actuel.
6. Conclusion « on change / on ne change pas », puis test officiel utilisé **une seule fois** pour le modèle retenu.

Le jeu de test officiel ne sert à rien avant l'étape 6.

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import scipy
import sklearn
import xgboost
from scipy import stats
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, GridSearchCV, RandomizedSearchCV
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5        # fenetre glissante, cf. notebook 03
PLAFOND_RUL = 125  # plafond de la RUL, cf. Phase 2
GRAINE = 42        # graine unique pour tout ce qui est aleatoire dans la recherche

# versions des bibliotheques, a noter dans le rapport (protocole 1.6)
print(f"numpy {np.__version__}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}, "
      f"xgboost {xgboost.__version__}, optuna {optuna.__version__}, scipy {scipy.__version__}")

In [ ]:
# --- Chargement et preparation : identique au notebook 03 (Phases 1 et 2) ---
train = load_fd001("train")  # fichier d'entrainement FD001

sensor_cols = get_sensor_columns(train.columns)  # les 21 capteurs bruts
constant_sensors = find_constant_sensors(train, sensor_cols)  # capteurs constants (Phase 1)
train = train.drop(columns=constant_sensors)  # on les retire
sensor_cols_restants = get_sensor_columns(train.columns)  # les 15 capteurs informatifs

# NB (point de methode signale) : le scaler est ajuste sur TOUT le train, avant les plis, comme dans le
# notebook 03. Sans effet sur XGBoost (la normalisation min-max garde l'ordre des valeurs, les arbres
# ne regardent que l'ordre), mais a refaire dans chaque pli pour un LSTM (protocole 1.3).
train, scaler = normalize_minmax(train, sensor_cols_restants)
train = lisser_capteurs(train, sensor_cols_restants, fenetre=FENETRE)
train = calculer_rul(train)
train = plafonner_rul(train, plafond=PLAFOND_RUL)


def ajouter_features_glissantes(df, sensor_cols, fenetre):
    """
    Ajoute, pour chaque capteur, la moyenne et l'ecart-type des `fenetre` derniers cycles,
    moteur par moteur (causal). Recopie de la fonction du notebook 03.

    Parameters
    ----------
    df : pandas.DataFrame
        Le tableau source (une copie est renvoyee).
    sensor_cols : list de str
        Les capteurs pour lesquels calculer les features.
    fenetre : int
        Taille de la fenetre glissante, en cycles.

    Returns
    -------
    pandas.DataFrame
        Le tableau avec "<capteur>_glissante_moy" et "<capteur>_glissante_std" en plus.
    """
    df = df.copy()  # copie independante

    for sensor_name in sensor_cols:  # un capteur a la fois
        colonne_moyenne = sensor_name + "_glissante_moy"
        colonne_ecart_type = sensor_name + "_glissante_std"
        df[colonne_moyenne] = 0.0  # valeurs provisoires
        df[colonne_ecart_type] = 0.0

        for moteur_id in df["unit_number"].unique():  # un moteur a la fois
            masque = df["unit_number"] == moteur_id  # Vrai sur les lignes de ce moteur
            # rolling() est causal et ne porte que sur ce moteur : pas de fuite
            df.loc[masque, colonne_moyenne] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).mean()
            df.loc[masque, colonne_ecart_type] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).std()

    return df


train = ajouter_features_glissantes(train, sensor_cols_restants, FENETRE)

# noms des colonnes moyenne / ecart-type glissants, par boucle explicite
colonnes_moyenne = []
colonnes_ecart_type = []
for colonne in train.columns:
    if colonne.endswith("_glissante_moy"):
        colonnes_moyenne.append(colonne)
    if colonne.endswith("_glissante_std"):
        colonnes_ecart_type.append(colonne)

train[colonnes_ecart_type] = train[colonnes_ecart_type].fillna(0.0)  # 1er cycle : ecart-type indefini -> 0

# 15 capteurs + 15 moyennes + 15 ecarts-types + time_cycles = 46 features (notebook 03)
feature_cols = sensor_cols_restants + colonnes_moyenne + colonnes_ecart_type + ["time_cycles"]
X = train[feature_cols]
y = train["RUL_cappee"]
groups = train["unit_number"]  # sert UNIQUEMENT a regrouper les lignes par moteur dans les plis

assert len(feature_cols) == 46
print(f"X : {X.shape}, y : {y.shape}, moteurs : {groups.nunique()}")

In [ ]:
def score_nasa(y_true, y_pred):
    """
    Score NASA (Saxena et al., 2008), asymetrique : une surestimation de la RUL (dangereuse)
    coute plus cher qu'une sous-estimation. Recopie du notebook 03.

    Parameters
    ----------
    y_true : array-like
        Les vraies RUL.
    y_pred : array-like
        Les RUL predites.

    Returns
    -------
    float
        La SOMME des penalites (diviser par le nombre de lignes pour la moyenne par prediction).
    """
    total = 0.0  # penalite cumulee

    for vrai, predit in zip(y_true, y_pred):  # un exemple a la fois
        d = predit - vrai  # positif = surestimation
        if d < 0:  # sous-estimation : penalite douce
            total += np.exp(-d / 13) - 1
        else:  # surestimation : penalite severe
            total += np.exp(d / 10) - 1

    return total


# sanity check identique au notebook 03
assert abs(score_nasa([100], [120]) - (np.exp(2) - 1)) < 0.01
assert abs(score_nasa([100], [80]) - (np.exp(20 / 13) - 1)) < 0.01

# --- Plis par moteur, calcules UNE fois (protocole 1.1 et 1.2) : ce sont les "plis de recherche" ---
plis = list(GroupKFold(n_splits=5, shuffle=True, random_state=GRAINE).split(X, y, groups=groups))

for idx_train, idx_val in plis:  # un moteur n'est jamais des deux cotes
    moteurs_train = set(groups.iloc[idx_train])
    moteurs_val = set(groups.iloc[idx_val])
    assert len(moteurs_train & moteurs_val) == 0, "un moteur est present des deux cotes !"
print("Plis de recherche : 5 plis, aucun moteur partage entre train et validation.")

In [ ]:
REGLAGES_ACTUELS = {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.05}  # Phase 3, jamais cherches


def evaluer_reglage(params, plis_a_utiliser):
    """
    Evalue un jeu d'hyperparametres XGBoost par validation croisee sur les plis donnes.

    Parameters
    ----------
    params : dict
        Hyperparametres passes a XGBRegressor (random_state=42 est ajoute ici).
    plis_a_utiliser : list de (idx_train, idx_val)
        Les plis par moteur a utiliser (jamais un decoupage aleatoire ligne par ligne).

    Returns
    -------
    dict
        Listes par pli : "rmse", "nasa_moyen" (score NASA moyen par prediction),
        "surestimes_20" (% de lignes surestimees de plus de 20 cycles).
    """
    rmse_plis = []
    nasa_plis = []
    surest_plis = []

    for idx_train, idx_val in plis_a_utiliser:  # un pli a la fois
        modele = XGBRegressor(random_state=GRAINE, **params)  # modele neuf a chaque pli
        modele.fit(X.iloc[idx_train], y.iloc[idx_train])
        y_val = y.iloc[idx_val].values
        y_pred = modele.predict(X.iloc[idx_val])

        rmse_plis.append(np.sqrt(mean_squared_error(y_val, y_pred)))
        nasa_plis.append(score_nasa(y_val, y_pred) / len(y_val))  # moyenne par prediction
        surest_plis.append(100.0 * np.sum(y_pred - y_val > 20) / len(y_val))  # en %

    return {"rmse": rmse_plis, "nasa_moyen": nasa_plis, "surestimes_20": surest_plis}


debut = time.time()
ref = evaluer_reglage(REGLAGES_ACTUELS, plis)
duree_ref = time.time() - debut

print(f"Reglages actuels : RMSE val = {np.mean(ref['rmse']):.2f} (ecart-type entre plis {np.std(ref['rmse']):.2f})")
print(f"                   score NASA moyen/prediction = {np.mean(ref['nasa_moyen']):.2f}  "
      f"(somme/pli ~ {np.mean(ref['nasa_moyen']) * len(y) / 5:.0f})")
print(f"                   surestimes > 20 cycles = {np.mean(ref['surestimes_20']):.1f} % des lignes")
print(f"Duree d'un essai (5 entrainements) : {duree_ref:.1f} s")

# sanity check : on doit retrouver le RMSE de validation du notebook 03 (16.86, tolerance de reproductibilite XGBoost)
assert abs(np.mean(ref["rmse"]) - 16.86) < 0.05, "ne redonne pas le 16.86 du notebook 03"

## Règle de décision (écrite le 2026-11-04, avant tout résultat de recherche)

**Référence** : réglages actuels `n_estimators=200, max_depth=5, learning_rate=0.05`, RMSE de validation 16.86 sur les plis de recherche (graine 42), score NASA moyen par prédiction 7.24, 11.6 % de lignes surestimées de plus de 20 cycles.

**Budget** : 60 essais par méthode (1 essai = 1 jeu d'hyperparamètres = 5 entraînements, un par pli), donc 300 entraînements par méthode.

**Étape 1 — Choisir un candidat** : parmi les trois méthodes, le candidat est l'essai au **RMSE moyen de validation le plus bas** sur les plis de recherche, parmi les essais dont le score NASA moyen n'est pas plus de 5 % au-dessus de la référence et dont le % de surestimations > 20 cycles n'est pas plus de 5 % (relatif) au-dessus de la référence. Un seul candidat est retenu pour la suite : on ne teste pas plusieurs candidats (sinon la p-value serait gonflée par les tests multiples).

**Étape 2 — Confirmer sur des plis neufs** : les plis de recherche ont servi à choisir, ils sont donc biaisés en faveur du candidat (biais d'optimisme). La confirmation se fait sur **10 découpages par moteur avec des graines jamais utilisées** (1000 à 1009), 5 plis chacun, référence et candidat évalués sur exactement les mêmes plis. On obtient 10 différences de RMSE (une par graine, moyenne des 5 plis).

**Étape 3 — On change les réglages si et seulement si les quatre conditions sont réunies** :
1. gain moyen de RMSE (référence − candidat) **≥ 0,3 cycle** (≈ 2 % du RMSE ; en dessous, la différence n'a pas d'intérêt pratique même si elle est « significative ») ;
2. test t apparié sur les 10 différences : **p < 0,05** ;
3. test de Wilcoxon apparié sur les 10 différences : **p < 0,05** ;
4. le score NASA moyen par prédiction et le % de surestimations > 20 cycles du candidat ne sont pas dégradés de plus de 5 % (relatif) par rapport à la référence, en moyenne sur les plis neufs.

Sinon : **on ne change pas** les réglages. À performance statistiquement équivalente, on garde le modèle le plus simple (règle 9 d'`AGENTS.md`) : si le candidat est équivalent mais plus simple (moins d'arbres, moins profond), on le signale sans le retenir automatiquement.

**Limites assumées** : les 5 plis d'une même graine ne sont pas indépendants (leurs jeux d'entraînement se recouvrent), donc le test porte sur les 10 moyennes par graine, pas sur 50 plis ; ces 10 graines se recoupent aussi (mêmes 100 moteurs) et la p-value reste un peu optimiste. Le test officiel (100 moteurs, un point par moteur) n'est utilisé qu'une fois, après la décision, pour rapporter le modèle retenu.

## Espace de recherche (fixé avant de lancer les recherches)

Budget identique pour les trois méthodes : **60 essais**, soit 300 entraînements chacune.

**Grid search** : grille de 3 hyperparamètres seulement (les trois déjà réglés à la main en Phase 3), 5 × 3 × 4 = 60 combinaisons, tout le reste aux valeurs par défaut de XGBoost. Une grille complète sur 7 hyperparamètres serait hors de portée du budget : c'est précisément la limite de la méthode (protocole §6 : grille pour 1 ou 2 hyperparamètres).

| Hyperparamètre | Grille (grid) | Espace (random search et Optuna) | Pourquoi cette plage |
|---|---|---|---|
| `max_depth` | 2, 3, 4, 5, 6 | entier 2 à 8 | courbe de validation de la Phase 3 : minimum vers 3, sur-apprentissage croissant au-delà de 6 |
| `learning_rate` | 0.02, 0.05, 0.1 | log-uniforme 0.01 à 0.3 | jamais exploré ; échelle logarithmique car 0.01 → 0.02 compte autant que 0.15 → 0.3 |
| `n_estimators` | 100, 200, 300, 400 | entier 50 à 600 | minimum de validation vers 90 arbres à `learning_rate=0.05` ; le bon nombre d'arbres dépend du `learning_rate` |
| `subsample` | défaut (1.0) | 0.5 à 1.0 | fraction de lignes par arbre : régularisation |
| `colsample_bytree` | défaut (1.0) | 0.3 à 1.0 | fraction de features par arbre : régularisation, utile avec 46 features très corrélées |
| `min_child_weight` | défaut (1) | log-uniforme 1 à 30 | taille minimale d'une feuille : limite le sur-apprentissage |
| `reg_lambda` | défaut (1) | log-uniforme 0.1 à 30 | pénalité L2 sur les poids des feuilles |

L'espace de random search et d'Optuna **contient** les valeurs de la grille et les réglages actuels : les trois méthodes cherchent dans le même univers, la grille en explore juste une tranche.

Pour chaque essai on enregistre : RMSE moyen de validation (objectif), score NASA moyen par prédiction, % de surestimations > 20 cycles, durée.

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import make_scorer

N_ESSAIS = 60  # budget identique pour les trois methodes (60 jeux d'hyperparametres, x5 plis)

# --- Grille (GridSearchCV) : 3 hyperparametres, 5 x 3 x 4 = 60 combinaisons ---
grille = {
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.02, 0.05, 0.1],
    "n_estimators": [100, 200, 300, 400],
}
nb_combinaisons = 1
for nom_hp, valeurs in grille.items():  # produit du nombre de valeurs de chaque hyperparametre
    nb_combinaisons = nb_combinaisons * len(valeurs)
assert nb_combinaisons == N_ESSAIS, f"la grille a {nb_combinaisons} combinaisons, pas {N_ESSAIS}"

# --- Distributions (RandomizedSearchCV) : memes bornes que celles utilisees par Optuna plus bas ---
# randint(a, b) tire dans [a, b-1] ; uniform(loc, scale) tire dans [loc, loc+scale] ; loguniform(a, b) dans [a, b]
distributions = {
    "max_depth": randint(2, 9),               # 2 a 8
    "learning_rate": loguniform(0.01, 0.3),
    "n_estimators": randint(50, 601),         # 50 a 600
    "subsample": uniform(0.5, 0.5),           # 0.5 a 1.0
    "colsample_bytree": uniform(0.3, 0.7),    # 0.3 a 1.0
    "min_child_weight": loguniform(1, 30),
    "reg_lambda": loguniform(0.1, 30),
}


def surestimes_20_pourcent(y_true, y_pred):
    """
    Pourcentage de lignes dont la RUL est surestimee de plus de 20 cycles.

    Parameters
    ----------
    y_true : array-like
        Les vraies RUL.
    y_pred : array-like
        Les RUL predites.

    Returns
    -------
    float
        Pourcentage entre 0 et 100.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 100.0 * np.sum(y_pred - y_true > 20) / len(y_true)


def nasa_moyen(y_true, y_pred):
    """
    Score NASA moyen par prediction (somme divisee par le nombre de lignes).

    Parameters
    ----------
    y_true : array-like
        Les vraies RUL.
    y_pred : array-like
        Les RUL predites.

    Returns
    -------
    float
        Penalite moyenne par prediction.
    """
    return score_nasa(y_true, y_pred) / len(y_true)


# scikit-learn MAXIMISE un score : pour une erreur, greater_is_better=False renvoie l'oppose (valeurs negatives)
scoring = {
    "rmse": make_scorer(lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)), greater_is_better=False),
    "nasa_moyen": make_scorer(nasa_moyen, greater_is_better=False),
    "surestimes_20": make_scorer(surestimes_20_pourcent, greater_is_better=False),
}
print(f"Grille : {nb_combinaisons} combinaisons. Random search et Optuna : {N_ESSAIS} essais sur {len(distributions)} hyperparametres.")

In [ ]:
import os
import pickle

DOSSIER_CACHE = ".cache_04d"  # resultats des recherches (longues) : evite de tout relancer si on reouvre le notebook
os.makedirs(DOSSIER_CACHE, exist_ok=True)


def charger_ou_calculer(nom, fonction):
    """
    Renvoie le resultat mis en cache s'il existe, sinon l'calcule avec `fonction`, le sauvegarde et le renvoie.

    Parameters
    ----------
    nom : str
        Nom du fichier de cache (sans extension).
    fonction : callable
        Fonction sans argument qui calcule le resultat (une recherche, longue).

    Returns
    -------
    object
        Le resultat (ici : un tableau pandas des essais et la duree en secondes).
    """
    chemin = os.path.join(DOSSIER_CACHE, nom + ".pkl")
    if os.path.exists(chemin):  # deja calcule lors d'une execution precedente
        with open(chemin, "rb") as f:
            return pickle.load(f)
    resultat = fonction()  # calcul long
    with open(chemin, "wb") as f:
        pickle.dump(resultat, f)
    return resultat


def resultats_sklearn_vers_tableau(recherche, nom_methode):
    """
    Transforme recherche.cv_results_ (GridSearchCV ou RandomizedSearchCV) en tableau, une ligne par essai.

    Parameters
    ----------
    recherche : GridSearchCV ou RandomizedSearchCV
        Une recherche deja entrainee (multi-metriques, scoring negatif).
    nom_methode : str
        Nom ecrit dans la colonne "methode".

    Returns
    -------
    pandas.DataFrame
        Colonnes : methode, essai, params, rmse, rmse_std, nasa_moyen, surestimes_20.
    """
    res = recherche.cv_results_
    lignes = []
    for i in range(len(res["params"])):  # un essai a la fois
        lignes.append({
            "methode": nom_methode,
            "essai": i + 1,
            "params": res["params"][i],
            "rmse": -res["mean_test_rmse"][i],  # on remet le signe : le scorer renvoyait l'oppose
            "rmse_std": res["std_test_rmse"][i],
            "nasa_moyen": -res["mean_test_nasa_moyen"][i],
            "surestimes_20": -res["mean_test_surestimes_20"][i],
        })
    return pd.DataFrame(lignes)


def lancer_grid():
    """
    Lance GridSearchCV sur la grille (60 combinaisons x 5 plis par moteur passes en cv=).

    Returns
    -------
    (pandas.DataFrame, float)
        Le tableau des essais et la duree totale en secondes.
    """
    debut_grid = time.time()
    recherche = GridSearchCV(
        XGBRegressor(random_state=GRAINE),
        param_grid=grille,
        cv=plis,              # LES plis par moteur (liste de (idx_train, idx_val)), jamais cv=5
        scoring=scoring,
        refit=False,          # on ne reentraine pas : seul le classement des essais nous interesse
        n_jobs=1,             # XGBoost parallelise deja en interne
    )
    recherche.fit(X, y)
    return resultats_sklearn_vers_tableau(recherche, "Grid"), time.time() - debut_grid


essais_grid, duree_grid = charger_ou_calculer("grid", lancer_grid)
print(f"GridSearchCV : {len(essais_grid)} essais en {duree_grid / 60:.1f} min")
print(essais_grid.sort_values("rmse").head(3)[["params", "rmse", "nasa_moyen", "surestimes_20"]].to_string())

In [ ]:
def lancer_random():
    """
    Lance RandomizedSearchCV : 60 tirages au hasard dans les distributions, memes plis par moteur.

    Returns
    -------
    (pandas.DataFrame, float)
        Le tableau des essais et la duree totale en secondes.
    """
    debut_random = time.time()
    recherche = RandomizedSearchCV(
        XGBRegressor(random_state=GRAINE),
        param_distributions=distributions,
        n_iter=N_ESSAIS,       # meme budget que la grille
        cv=plis,               # memes plis par moteur
        scoring=scoring,
        refit=False,
        random_state=GRAINE,   # tirages reproductibles
        n_jobs=1,
    )
    recherche.fit(X, y)
    return resultats_sklearn_vers_tableau(recherche, "Random"), time.time() - debut_random


essais_random, duree_random = charger_ou_calculer("random", lancer_random)
print(f"RandomizedSearchCV : {len(essais_random)} essais en {duree_random / 60:.1f} min")
print(essais_random.sort_values("rmse").head(3)[["params", "rmse", "nasa_moyen", "surestimes_20"]].to_string())

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)  # pas un message par essai


def objectif(essai):
    """
    Fonction objectif d'Optuna : propose des hyperparametres, renvoie le RMSE moyen de validation.

    Le score NASA moyen et le % de surestimations sont gardes en attributs de l'essai (pas optimises).

    Parameters
    ----------
    essai : optuna.trial.Trial
        L'essai en cours ; c'est lui qui propose les valeurs via suggest_*.

    Returns
    -------
    float
        RMSE moyen sur les 5 plis par moteur (a minimiser).
    """
    params = {
        "max_depth": essai.suggest_int("max_depth", 2, 8),
        "learning_rate": essai.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": essai.suggest_int("n_estimators", 50, 600),
        "subsample": essai.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": essai.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": essai.suggest_float("min_child_weight", 1, 30, log=True),
        "reg_lambda": essai.suggest_float("reg_lambda", 0.1, 30, log=True),
    }
    resultat = evaluer_reglage(params, plis)  # memes plis par moteur, meme fonction que pour la reference
    essai.set_user_attr("rmse_std", float(np.std(resultat["rmse"])))
    essai.set_user_attr("nasa_moyen", float(np.mean(resultat["nasa_moyen"])))
    essai.set_user_attr("surestimes_20", float(np.mean(resultat["surestimes_20"])))
    return float(np.mean(resultat["rmse"]))


def lancer_optuna():
    """
    Lance l'etude Optuna (TPE, graine 42, 60 essais).

    Returns
    -------
    ((pandas.DataFrame, optuna.study.Study), float)
        Le tableau des essais, l'etude (pour les figures) et la duree totale en secondes.
    """
    debut_optuna = time.time()
    etude = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=GRAINE))
    etude.optimize(objectif, n_trials=N_ESSAIS)

    lignes = []
    for t in etude.trials:  # un essai a la fois, dans l'ordre
        lignes.append({
            "methode": "Optuna",
            "essai": t.number + 1,
            "params": dict(t.params),
            "rmse": t.value,
            "rmse_std": t.user_attrs["rmse_std"],
            "nasa_moyen": t.user_attrs["nasa_moyen"],
            "surestimes_20": t.user_attrs["surestimes_20"],
        })
    return (pd.DataFrame(lignes), etude), time.time() - debut_optuna


(essais_optuna, etude_optuna), duree_optuna = charger_ou_calculer("optuna", lancer_optuna)
print(f"Optuna : {len(essais_optuna)} essais en {duree_optuna / 60:.1f} min")
print(essais_optuna.sort_values("rmse").head(3)[["params", "rmse", "nasa_moyen", "surestimes_20"]].to_string())

In [ ]:
DUREES = {"Grid": duree_grid, "Random": duree_random, "Optuna": duree_optuna}
tous_les_essais = pd.concat([essais_grid, essais_random, essais_optuna], ignore_index=True)  # 180 essais

# reference : reglages actuels sur les memes plis (cellule 5)
ref_rmse = float(np.mean(ref["rmse"]))
ref_nasa = float(np.mean(ref["nasa_moyen"]))
ref_surest = float(np.mean(ref["surestimes_20"]))

# --- Tableau des methodes : meilleur essai en RMSE de chaque methode ---
print(f"Reference (reglages actuels) : RMSE {ref_rmse:.3f} | NASA moyen {ref_nasa:.3f} | surestimes>20 {ref_surest:.2f} %\n")
for methode in ["Grid", "Random", "Optuna"]:
    sous_tableau = tous_les_essais[tous_les_essais["methode"] == methode]
    meilleur = sous_tableau.sort_values("rmse").iloc[0]  # essai au RMSE le plus bas
    print(f"{methode:7s} budget={len(sous_tableau)} essais ({len(sous_tableau) * 5} entrainements), duree={DUREES[methode] / 60:.1f} min")
    print(f"        meilleur RMSE={meilleur['rmse']:.3f} (gain {ref_rmse - meilleur['rmse']:+.3f} vs reference), "
          f"NASA moyen={meilleur['nasa_moyen']:.3f}, surestimes>20={meilleur['surestimes_20']:.2f} %")
    params_arrondis = {}
    for nom_hp, valeur in meilleur["params"].items():  # arrondi pour la lecture
        if isinstance(valeur, float):
            params_arrondis[nom_hp] = round(valeur, 4)
        else:
            params_arrondis[nom_hp] = valeur
    print(f"        params={params_arrondis}\n")

# --- Candidat selon la regle (ecrite avant les resultats) : essais "eligibles" puis RMSE le plus bas ---
eligible = (tous_les_essais["nasa_moyen"] <= 1.05 * ref_nasa) & (tous_les_essais["surestimes_20"] <= 1.05 * ref_surest)
print(f"Essais eligibles (NASA et surestimations pas plus de 5 % au-dessus de la reference) : {int(eligible.sum())} sur {len(tous_les_essais)}")
candidat = tous_les_essais[eligible].sort_values("rmse").iloc[0]
PARAMS_CANDIDAT = dict(candidat["params"])  # un seul candidat est retenu pour la suite
print(f"Candidat : methode={candidat['methode']}, essai {candidat['essai']}, RMSE={candidat['rmse']:.3f}, "
      f"gain sur les plis de recherche = {ref_rmse - candidat['rmse']:+.3f}")
print(PARAMS_CANDIDAT)

# bruit entre plis, pour situer le gain : rappel de l'ecart-type de la reference
print(f"\nEcart-type du RMSE entre plis (reference) : {np.std(ref['rmse']):.2f} -> le gain est {(ref_rmse - candidat['rmse']) / np.std(ref['rmse']):.2f} fois ce bruit")

In [ ]:
COULEURS = {"Grid": "tab:blue", "Random": "tab:orange", "Optuna": "tab:green"}

# --- Figure 1 : meilleur RMSE atteint apres k essais (a budget egal, quelle methode descend le plus vite ?) ---
fig, ax = plt.subplots(figsize=(8, 4.8))
for methode in ["Grid", "Random", "Optuna"]:
    sous_tableau = tous_les_essais[tous_les_essais["methode"] == methode].sort_values("essai")
    meilleur_jusqu_ici = []  # meilleur RMSE vu apres chaque essai
    courant = np.inf
    for valeur in sous_tableau["rmse"]:
        if valeur < courant:
            courant = valeur
        meilleur_jusqu_ici.append(courant)
    ax.plot(sous_tableau["essai"], meilleur_jusqu_ici, label=methode, color=COULEURS[methode], linewidth=2)
    ax.scatter(sous_tableau["essai"], sous_tableau["rmse"], s=10, color=COULEURS[methode], alpha=0.35)  # chaque essai
ax.axhline(ref_rmse, linestyle="--", color="grey", label=f"reglages actuels ({ref_rmse:.2f})")
ax.set_ylim(16.3, 17.8)  # on zoome : tout se joue sur quelques dixiemes
ax.set_xlabel("nombre d'essais (1 essai = 5 entrainements)")
ax.set_ylabel("RMSE de validation (cycles)")
ax.set_title("Meilleur RMSE selon le budget, trois methodes (points = chaque essai)")
ax.legend()
plt.tight_layout()
plt.savefig("../docs/figures/04d_historique_recherche.png", dpi=120)
plt.show()

# --- Figure 2 : importance des hyperparametres (etude Optuna, methode fANOVA) ---
importances = optuna.importance.get_param_importances(etude_optuna)  # dict nom -> part de la variance du RMSE
noms_hp = list(importances.keys())
valeurs_hp = list(importances.values())
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(noms_hp, valeurs_hp, color="tab:green")
ax.invert_yaxis()  # le plus important en haut
for position, valeur in enumerate(valeurs_hp):
    ax.text(valeur + 0.005, position, f"{valeur:.2f}", va="center")
ax.set_xlabel("part de la variation du RMSE expliquee (fANOVA, 60 essais Optuna)")
ax.set_title("Importance des hyperparametres")
plt.tight_layout()
plt.savefig("../docs/figures/04d_importance_hyperparametres.png", dpi=120)
plt.show()
print({nom_hp: round(valeur, 3) for nom_hp, valeur in importances.items()})

# --- Figure 3 : RMSE contre score NASA moyen, un point par essai (equilibre demande par le cahier des charges) ---
fig, ax = plt.subplots(figsize=(7, 5))
for methode in ["Grid", "Random", "Optuna"]:
    sous_tableau = tous_les_essais[tous_les_essais["methode"] == methode]
    ax.scatter(sous_tableau["rmse"], sous_tableau["nasa_moyen"], s=18, alpha=0.6, label=methode, color=COULEURS[methode])
ax.scatter([ref_rmse], [ref_nasa], marker="*", s=200, color="black", label="reglages actuels", zorder=5)
ax.set_xlim(16.3, 18.0)
ax.set_ylim(6.0, 8.0)
ax.set_xlabel("RMSE de validation (cycles)")
ax.set_ylabel("score NASA moyen par prediction")
ax.set_title("RMSE et score NASA par essai : les deux baissent ensemble ?")
ax.legend()
plt.tight_layout()
plt.savefig("../docs/figures/04d_rmse_vs_score_nasa.png", dpi=120)
plt.show()

correlation = np.corrcoef(tous_les_essais["rmse"], tous_les_essais["nasa_moyen"])[0, 1]
print(f"Correlation RMSE / score NASA moyen sur les 180 essais : {correlation:.2f}")

In [ ]:
GRAINES_CONFIRMATION = list(range(1000, 1010))  # 10 graines jamais utilisees dans la recherche (qui utilisait 42)


def lancer_confirmation():
    """
    Evalue reference et candidat sur 10 decoupages par moteur neufs (5 plis chacun).

    Pour chaque graine, les DEUX modeles sont evalues sur exactement les memes plis (comparaison appariee).

    Returns
    -------
    dict
        {"reference": [...], "candidat": [...]} : pour chaque graine, le dict renvoye par evaluer_reglage.
    """
    resultats = {"reference": [], "candidat": []}
    for graine in GRAINES_CONFIRMATION:  # un nouveau decoupage par moteur a chaque graine
        plis_neufs = list(GroupKFold(n_splits=5, shuffle=True, random_state=graine).split(X, y, groups=groups))
        for idx_train, idx_val in plis_neufs:  # un moteur n'est jamais des deux cotes
            assert len(set(groups.iloc[idx_train]) & set(groups.iloc[idx_val])) == 0
        resultats["reference"].append(evaluer_reglage(REGLAGES_ACTUELS, plis_neufs))
        resultats["candidat"].append(evaluer_reglage(PARAMS_CANDIDAT, plis_neufs))
    return resultats


confirmation = charger_ou_calculer("confirmation", lancer_confirmation)

# une valeur par graine = moyenne des 5 plis (les 5 plis d'une graine ne sont pas independants : on teste ces 10 moyennes)
def moyennes_par_graine(nom_modele, nom_metrique):
    """
    Moyenne sur les 5 plis de `nom_metrique`, pour chacune des 10 graines.

    Parameters
    ----------
    nom_modele : str
        "reference" ou "candidat".
    nom_metrique : str
        "rmse", "nasa_moyen" ou "surestimes_20".

    Returns
    -------
    numpy.ndarray
        10 valeurs, une par graine.
    """
    valeurs = []
    for resultat_graine in confirmation[nom_modele]:  # une graine a la fois
        valeurs.append(np.mean(resultat_graine[nom_metrique]))
    return np.array(valeurs)


rmse_ref = moyennes_par_graine("reference", "rmse")
rmse_cand = moyennes_par_graine("candidat", "rmse")
nasa_ref = moyennes_par_graine("reference", "nasa_moyen")
nasa_cand = moyennes_par_graine("candidat", "nasa_moyen")
surest_ref = moyennes_par_graine("reference", "surestimes_20")
surest_cand = moyennes_par_graine("candidat", "surestimes_20")

gains = rmse_ref - rmse_cand  # positif = le candidat est meilleur
print("Graine | RMSE reference | RMSE candidat | gain")
for i, graine in enumerate(GRAINES_CONFIRMATION):
    print(f" {graine}  |     {rmse_ref[i]:6.3f}     |    {rmse_cand[i]:6.3f}     | {gains[i]:+.3f}")

print(f"\nPlis de recherche (graine 42, utilises pour CHOISIR) : gain = {ref_rmse - candidat['rmse']:+.3f}")
print(f"Plis neufs (10 graines)                             : gain moyen = {gains.mean():+.3f} (ecart-type entre graines {gains.std(ddof=1):.3f}, "
      f"{int(np.sum(gains > 0))}/10 graines favorables)")
print(f"-> ecart attribuable au biais d'optimisme : {(ref_rmse - candidat['rmse']) - gains.mean():+.3f}")

# --- Tests apparies sur les 10 differences ---
test_t = stats.ttest_rel(rmse_ref, rmse_cand)  # test t apparie
test_w = stats.wilcoxon(rmse_ref, rmse_cand)   # Wilcoxon apparie (sans supposer une loi normale)
print(f"\nTest t apparie : t={test_t.statistic:.2f}, p={test_t.pvalue:.4f}")
print(f"Wilcoxon apparie : statistique={test_w.statistic:.1f}, p={test_w.pvalue:.4f}")

# --- Les 4 conditions de la regle de decision (ecrite avant les resultats) ---
cond_gain = gains.mean() >= 0.3
cond_t = test_t.pvalue < 0.05
cond_w = test_w.pvalue < 0.05
nasa_ok = nasa_cand.mean() <= 1.05 * nasa_ref.mean()
surest_ok = surest_cand.mean() <= 1.05 * surest_ref.mean()
cond_garde_fous = nasa_ok and surest_ok

print(f"\nScore NASA moyen : reference {nasa_ref.mean():.3f}, candidat {nasa_cand.mean():.3f} ({100 * (nasa_cand.mean() / nasa_ref.mean() - 1):+.1f} %)")
print(f"Surestimes > 20 cycles : reference {surest_ref.mean():.2f} %, candidat {surest_cand.mean():.2f} %")
print(f"\n1. gain moyen >= 0.3        : {cond_gain} ({gains.mean():+.3f})")
print(f"2. test t p < 0.05          : {cond_t} (p={test_t.pvalue:.4f})")
print(f"3. Wilcoxon p < 0.05        : {cond_w} (p={test_w.pvalue:.4f})")
print(f"4. garde-fous NASA / surest.: {cond_garde_fous}")
DECISION_CHANGER = cond_gain and cond_t and cond_w and cond_garde_fous
print(f"\n=> DECISION : {'ON CHANGE les reglages' if DECISION_CHANGER else 'ON NE CHANGE PAS les reglages'}")

In [ ]:
# Figure : gain de RMSE du candidat sur chaque decoupage neuf, compare au seuil de la regle et au gain des plis de recherche
fig, ax = plt.subplots(figsize=(8.5, 4.8))
positions = np.arange(len(GRAINES_CONFIRMATION))
ax.bar(positions, gains, color="tab:green", alpha=0.8, label="gain par graine (plis neufs)")
ax.axhline(gains.mean(), color="tab:green", linestyle="-", label=f"gain moyen sur plis neufs ({gains.mean():.2f})")
ax.axhline(ref_rmse - candidat["rmse"], color="tab:red", linestyle=":", label=f"gain sur les plis de recherche ({ref_rmse - candidat['rmse']:.2f}) : optimiste")
ax.axhline(0.3, color="black", linestyle="--", label="seuil de la regle (0.30)")
ax.axhline(0.0, color="grey", linewidth=0.8)
etiquettes = []
for graine in GRAINES_CONFIRMATION:
    etiquettes.append(str(graine))
ax.set_xticks(positions)
ax.set_xticklabels(etiquettes)
ax.set_xlabel("graine du decoupage par moteur")
ax.set_ylabel("gain de RMSE, reference - candidat (cycles)")
ax.set_title("Candidat Optuna contre reglages actuels, 10 decoupages neufs")
ax.set_ylim(-0.05, 0.5)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("../docs/figures/04d_confirmation_plis_neufs.png", dpi=120)
plt.show()

In [ ]:
# --- Test officiel : utilise UNE seule fois, pour le modele RETENU (les reglages actuels, decision de la cellule 14) ---
# Le candidat Optuna n'est volontairement PAS evalue sur le test : il n'est pas retenu, et le regarder
# permettrait de revenir sur la decision en fonction du test (protocole 1.4).
assert not DECISION_CHANGER, "la decision a change : le modele retenu n'est plus les reglages actuels"

test = load_fd001("test")  # fichier de test FD001
test = test.drop(columns=constant_sensors)  # memes capteurs retires que sur le train
test[sensor_cols_restants] = scaler.transform(test[sensor_cols_restants])  # transform seul, jamais de re-fit sur le test
test = lisser_capteurs(test, sensor_cols_restants, fenetre=FENETRE)
test = ajouter_features_glissantes(test, sensor_cols_restants, FENETRE)
test[colonnes_ecart_type] = test[colonnes_ecart_type].fillna(0.0)

# une ligne par moteur : son dernier cycle observe
indices_derniers_cycles = test.groupby("unit_number")["time_cycles"].idxmax()
dernier_cycle_par_moteur = test.loc[indices_derniers_cycles].sort_values("unit_number").reset_index(drop=True)
rul_test_vraie = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL_vraie"])
assert dernier_cycle_par_moteur.shape[0] == 100 and rul_test_vraie.shape[0] == 100

modele_retenu = XGBRegressor(random_state=GRAINE, **REGLAGES_ACTUELS)
modele_retenu.fit(X, y)  # entraine sur les 100 moteurs d'entrainement

y_test_vraie = rul_test_vraie["RUL_vraie"].values
y_test_pred = np.clip(modele_retenu.predict(dernier_cycle_par_moteur[feature_cols]), 0, None)  # RUL negative sans sens

rmse_test = np.sqrt(mean_squared_error(y_test_vraie, y_test_pred))
nasa_test = score_nasa(y_test_vraie, y_test_pred)
nb_surestimes_20 = int(np.sum(y_test_pred - y_test_vraie > 20))
print(f"Modele retenu (reglages actuels), test officiel : RMSE={rmse_test:.2f}, score NASA={nasa_test:.1f}, "
      f"surestimes>20={nb_surestimes_20}/100, score NASA moyen/moteur={nasa_test / 100:.2f}")
print("Phase 3 (docs/03_modelisation.md) : RMSE=19.70, score NASA=852.9, surestimes>20=9/100")
assert abs(rmse_test - 19.70) < 0.05, "ne redonne pas le 19.70 de la Phase 3"

In [ ]:
# Tableau recapitulatif des methodes (meme budget : 60 essais = 300 entrainements chacune)
lignes_recap = []
for methode in ["Grid", "Random", "Optuna"]:
    sous_tableau = tous_les_essais[tous_les_essais["methode"] == methode]
    meilleur = sous_tableau.sort_values("rmse").iloc[0]
    lignes_recap.append({
        "methode": methode,
        "hyperparametres cherches": 3 if methode == "Grid" else 7,
        "essais": len(sous_tableau),
        "duree (min)": round(DUREES[methode] / 60, 1),
        "meilleur RMSE": round(meilleur["rmse"], 3),
        "gain vs actuel (plis de recherche)": round(ref_rmse - meilleur["rmse"], 3),
        "NASA moyen": round(meilleur["nasa_moyen"], 3),
        "surestimes>20 (%)": round(meilleur["surestimes_20"], 2),
    })
recap = pd.DataFrame(lignes_recap)
print(recap.to_string(index=False))
print(f"\nReference (plis de recherche) : RMSE {ref_rmse:.3f}, NASA moyen {ref_nasa:.3f}, surestimes>20 {ref_surest:.2f} %")
print(f"Confirmation (10 graines neuves) : gain Optuna = {gains.mean():+.3f}, t p={test_t.pvalue:.4f}, Wilcoxon p={test_w.pvalue:.4f} -> decision : ne pas changer")

## Conclusion : on ne change pas les réglages

**Décision** : les réglages `n_estimators=200, max_depth=5, learning_rate=0.05` sont conservés.

**Pourquoi**
- La recherche (3 méthodes, 60 essais chacune, plis par moteur) trouve des réglages au RMSE plus bas : 16,50 contre 16,86 sur les plis de recherche (gain 0,36). Les trois méthodes arrivent à moins de 0,02 l'une de l'autre ; Optuna converge le plus vite (≈ 20 essais, contre ≈ 40 pour le random search).
- Ce gain est en partie un **biais d'optimisme** : sur 10 découpages par moteur jamais utilisés pour choisir (graines 1000 à 1009), le gain n'est plus que de **0,226 cycle** (10 graines sur 10 favorables ; test t apparié p = 0,0003 ; Wilcoxon p = 0,002). Environ 0,13 du gain apparent venait du choix du meilleur de 180 essais sur les mêmes plis.
- La différence est donc **réelle et significative, mais plus petite que le seuil de 0,3 cycle fixé avant les résultats** (≈ 1,3 % du RMSE, contre un écart-type de 1,25 entre plis). La règle (condition 1) s'applique : on ne remplace pas un modèle validé pour un gain pratiquement négligeable (règle 9 d'`AGENTS.md`).
- Aucun compromis RMSE / score NASA à arbitrer : les deux baissent ensemble (corrélation 0,87 sur 180 essais). Le score NASA moyen du candidat est 7 % plus bas sur les plis neufs, les surestimations de plus de 20 cycles restent stables (11,4 % contre 11,5 %).

**Ce que ça apprend**
- Le plancher d'environ 16,5 de RMSE de validation n'est pas un problème de réglage : une recherche sur 7 hyperparamètres avec trois méthodes différentes ne descend pas dessous. Pour l'abaisser, il faut d'autres features ou d'autres modèles (sections 5 et 7 du protocole), pas d'autres hyperparamètres.
- Hyperparamètre le plus influent : `learning_rate` (importance fANOVA 0,41), puis `max_depth` (0,16), `reg_lambda` (0,14), `n_estimators` (0,12).

**Limites**
- Les 10 graines de confirmation réutilisent les mêmes 100 moteurs : la p-value reste un peu optimiste, et le gain ne se transfère pas forcément à d'autres moteurs. Une validation croisée imbriquée serait le niveau au-dessus.
- Espace de recherche limité à 7 hyperparamètres XGBoost ; budget de 60 essais par méthode ; la grille n'en explore que 3.
- Une seule exécution de chaque recherche (graine 42) : le classement entre méthodes (écarts de 0,02) n'est pas significatif.
- Le test officiel (100 moteurs, un point par moteur) n'a servi qu'au modèle retenu ; le candidat n'y a pas été évalué.